In [1]:
import transformers
import datasets
import torch
import modelscope
print(transformers.__version__)
print(datasets.__version__)
print(torch.__version__)
print(modelscope.__version__)
%pip install simplejson

c:\Users\havei\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.23.1
3.6.0
2.2.1+cpu
1.29.2
Note: you may need to restart the kernel to use updated packages.


    Markdown (>=3.0<3.3) ; python_version < "3.6"
             ~~~~~~^

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


4.23.1
3.6.0
2.2.1+cpu
1.29.2

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchaudio
import numpy as np
import contextlib
import os
import torch
import torch.nn.functional as F
from pathlib import Path
import librosa
from xgboost import XGBClassifier
import random
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
from sklearn.metrics import accuracy_score, confusion_matrix
# from modelscope.models import Model
# from funasr import AutoModel


https://huggingface.co/emotion2vec/emotion2vec_base

In [3]:

class VoiceFolder(Dataset):
    def __init__(self, root, transform=None, sr=16000, target_len=48000):
        self.file_paths = []
        for path in Path(root).rglob("*.wav"):
            w, _ = torchaudio.load(path)
            w = w.squeeze(0)
            cur_len = w.shape[-1]
            self.file_paths.append(path)
            # else:
            # print("Drop:",path)
        self.transform = transform
        self.sr = sr
        self.target_len = target_len
        self.labels = sorted({p.parent.name for p in self.file_paths})
        self.label2id = {label: idx for idx, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.file_paths)

    def pad_or_trim(self, waveform):
        cur_len = waveform.shape[-1]
        if cur_len >= self.target_len:
            start = cur_len - self.target_len
            if(start == 0):
                return waveform[..., start:start + self.target_len]
            start = random.randrange(start)
            return waveform[..., start:start + self.target_len]
        else:
            pad_len = self.target_len - cur_len
            return F.pad(waveform, (0, pad_len))
        # 將音訊長度統整為target_len

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            waveform, file_sr = torchaudio.load(path)
            if file_sr != self.sr:
                resampler = torchaudio.transforms.Resample(
                    orig_freq=file_sr, new_freq=self.sr)
                waveform = resampler(waveform)
            waveform = waveform.squeeze(0)
            # 設定預設transform
            waveform = waveform.float()
            max_val = waveform.abs().max()
            if max_val > 0:
                waveform = waveform / max_val
            waveform = self.pad_or_trim(waveform)
            label_str = path.parent.name
            label = self.label2id.get(label_str)
            if self.transform:
                waveform = self.transform(waveform)
            return waveform, label
        except Exception as e:
            print(f"[ERROR] loading {path}: {e}")
            return torch.zeros(self.target_len), -1


def augment_waveform(waveform, sample_rate=16000):  # 資料強化transform

    # 隨機加入雜訊

    if random.randrange(4) == 0:
        noise = torch.randn_like(waveform) * 0.005  # 控制雜訊強度
        waveform = waveform + noise

    # 隨機變更音高（±2個半音）
    if random.randrange(5) == 4:
        try:
            n_steps = random.uniform(-2.0, 2.0)
            waveform = waveform.numpy()
            waveform = librosa.effects.pitch_shift(
                y=waveform, sr=float(sample_rate), n_steps=float(n_steps))
            waveform = torch.from_numpy(waveform)
        except Exception as e:
            print(f"[WARN] pitch shift failed: {e}")
            waveform = torch.from_numpy(waveform)
    return waveform


train_root = r"C:\Users\havei\Desktop\voice\Voice_Emotion_Classification\Train"  # 需動
test_root = r"C:\Users\havei\Desktop\voice\Voice_Emotion_Classification\Test" # 需動
batch_size = 32
sr = 16000

train_dataset_3s = VoiceFolder(
    root=train_root, transform=augment_waveform, sr=sr, target_len=sr*3)
test_dataset_3s = VoiceFolder(root=test_root, sr=sr, target_len=sr*3)

train_loader_3s = DataLoader(
    train_dataset_3s, batch_size=batch_size, shuffle=True)
val_loader_3s = DataLoader(
    test_dataset_3s, batch_size=batch_size, shuffle=True)

train_labels_set = set(train_dataset_3s.labels)
test_labels_set = set(test_dataset_3s.labels)
print(train_labels_set)
if train_labels_set != test_labels_set:
    print("⚠️ Train and test labels do not match!")
    print("Train labels only:", train_labels_set - test_labels_set)
    print("Test labels only :", test_labels_set - train_labels_set)
else:
    print("✅ Train and test labels match exactly.")


{'neutral', 'anger', 'disgust', 'sad', 'happy', 'fear'}
✅ Train and test labels match exactly.


In [ ]:
# --------------------------------------------------------
# 設定裝置
# --------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用裝置:", device)

# --------------------------------------------------------
# Emotion2Vec 初始化
# --------------------------------------------------------
feature_model = pipeline(
    task=Tasks.emotion_recognition,
    model="iic/emotion2vec_base"
)

# --------------------------------------------------------
# Feature cache 函數
# --------------------------------------------------------
def get_cached_feature(waveform, cache_dir, idx):
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"{idx}.npy")
    if os.path.exists(cache_file):
        feat_vec = torch.from_numpy(np.load(cache_file))
    else:
        waveform = waveform.unsqueeze(0)
        with torch.no_grad():
            with open(os.devnull, 'w') as fnull:
                with contextlib.redirect_stderr(fnull):
                    out = feature_model(waveform)
        feat = out[0]['feats']
        feat_vec = torch.from_numpy(feat).float()
        # 檢查 NaN / Inf
        if torch.isnan(feat_vec).any() or torch.isinf(feat_vec).any():
            raise RuntimeError(f"Feature {idx} contains NaN/Inf")
        np.save(cache_file, feat_vec.numpy())
    return feat_vec

# --------------------------------------------------------
# 提取 dataset 特徵並快取
# --------------------------------------------------------
def extract_dataset_features_with_cache(dataloader, cache_dir):
    all_features, all_labels = [], []
    for batch_idx, (waveforms, labels) in enumerate(dataloader):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        for i in range(waveforms.size(0)):
            try:
                feat_vec = get_cached_feature(waveforms[i], cache_dir, batch_idx*waveforms.size(0)+i)
                all_features.append(feat_vec)
                all_labels.append(labels[i].item())
            except RuntimeError as e:
                print(f"[WARN] {e}")
                continue
    if len(all_features) == 0:
        raise RuntimeError("無可用特徵")
    return torch.stack(all_features).numpy(), np.array(all_labels)

# --------------------------------------------------------
# 提取或載入特徵
# --------------------------------------------------------
train_cache_dir = "cache/train"
test_cache_dir = "cache/test"

print("提取 train 特徵...")
train_feats, train_labels = extract_dataset_features_with_cache(train_loader_3s, train_cache_dir)

print("提取 test 特徵...")
test_feats, test_labels = extract_dataset_features_with_cache(val_loader_3s, test_cache_dir)

print("Train:", train_feats.shape, "Labels:", train_labels.shape)
print("Test :", test_feats.shape, "Labels:", test_labels.shape)

# --------------------------------------------------------
# XGBClassifier 建立
# --------------------------------------------------------
num_classes = len(train_dataset_3s.labels)
model_file = "xgb_model.json"

if os.path.exists(model_file):
    print("載入已存在 XGBoost 模型...")
    model = XGBClassifier()
    model.load_model(model_file)
else:
    print("訓練 XGBoost 模型...")
    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="gpu_hist" if device == "cuda" else "hist"
    )
    model.fit(train_feats, train_labels)
    model.save_model(model_file)
    print("XGBoost 模型已儲存！")

# --------------------------------------------------------
# 訓練/測試準確率
# --------------------------------------------------------
train_pred_probs = model.predict_proba(train_feats)
train_preds = np.argmax(train_pred_probs, axis=1)
train_acc = accuracy_score(train_labels, train_preds)
print(f"訓練準確率: {train_acc*100:.2f}%")

test_pred_probs = model.predict_proba(test_feats)
test_preds = np.argmax(test_pred_probs, axis=1)
test_acc = accuracy_score(test_labels, test_preds)
print(f"測試準確率: {test_acc*100:.2f}%")

# 混淆矩陣
cm = confusion_matrix(test_labels, test_preds)
print("\n測試集混淆矩陣:")
print(cm)


使用裝置: cpu


2025-12-12 13:49:36,612 - modelscope - WARNING - Model revision not specified, use revision: v2.0.4


2025-12-12 13:49:44,563 - modelscope - WARNING - Model revision not specified, use revision: v2.0.4
2025-12-12 13:49:45,434 - modelscope - INFO - initiate model from C:\Users\havei\.cache\modelscope\hub\models\iic\emotion2vec_base
2025-12-12 13:49:45,434 - modelscope - INFO - initiate model from location C:\Users\havei\.cache\modelscope\hub\models\iic\emotion2vec_base.
2025-12-12 13:49:45,441 - modelscope - INFO - initialize model from C:\Users\havei\.cache\modelscope\hub\models\iic\emotion2vec_base


Notice: ffmpeg is not installed. torchaudio is used to load audio
If you want to use ffmpeg backend to load audio, please install it by:
	sudo apt install ffmpeg # ubuntu
	# brew install ffmpeg # mac
New version available: 1.2.7. Your current version is 1.1.0.
Please use the command "pip install -U funasr" to upgrade.


2025-12-12 13:49:55,575 - modelscope - WARNING - No preprocessor field found in cfg.
2025-12-12 13:49:55,583 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2025-12-12 13:49:55,583 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': 'C:\\Users\\havei\\.cache\\modelscope\\hub\\models\\iic\\emotion2vec_base'}. trying to build by task and model information.
2025-12-12 13:49:55,583 - modelscope - INFO - No preprocessor key ('funasr', 'emotion-recognition') found in PREPROCESSOR_MAP, skip building preprocessor. If the pipeline runs normally, please ignore this log.
2025-12-12 13:49:55,587 - modelscope - INFO - cuda is not available, using cpu instead.


 正在提取 train 特徵 


KeyboardInterrupt: 